# 01 — Reproduction: vanilla TabPFN on NSL-KDD

The baseline arm end to end: load, inspect, preprocess, fit one TabPFN context, score it.

No pipeline logic lives in this notebook. Every step calls the same tested package code the scripts run, so what you see here is what produced the numbers in `reports/`.

**Runtime:** about 2 minutes.

**Scale:** this is a *demo-scale* run — a 5,000-row context scored on 600 test rows. The headline results in `reports/` use a 10,000-row context and 5,000 test rows, which takes roughly 40 minutes on this machine. Section 6 puts the two side by side. The pipeline is identical; only the row counts differ.

In [ ]:
import time

import numpy as np
import pandas as pd

from tabpfn_nids import config
from tabpfn_nids.data_pipeline import load_nsl_kdd, load_and_preprocess_nsl_kdd
from tabpfn_nids.models import TabPFNWrapper
from tabpfn_nids.evaluation import compute_metrics, format_metrics, plot_all

# Demo-scale knobs. Cost is roughly linear in TEST_ROWS and superlinear in
# CONTEXT_ROWS: 5,000 context costs ~144s per 1,000 test rows on this M1,
# 10,000 context costs ~459s. Raise these to trade runtime for tighter metrics.
CONTEXT_ROWS = 5_000
TEST_ROWS = 600
N_ESTIMATORS = 2

config.set_seed(config.SEED)
notebook_started = time.perf_counter()

print(f'checkpoint : {config.TABPFN_CHECKPOINT}')
print(f'device     : {config.resolve_device()}')
print(f'seed       : {config.SEED}')

## 1. Load the raw data

NSL-KDD ships as headerless `.txt`. Column names come from the `@attribute` lines in the `KDDTrain+.arff` file in the same archive — read from the dataset rather than hard-coded from memory.

In [ ]:
train_df, test_df = load_nsl_kdd()

print(f'train : {train_df.shape[0]:,} rows x {train_df.shape[1]} columns')
print(f'test  : {test_df.shape[0]:,} rows x {test_df.shape[1]} columns')
train_df.head()

## 2. Dataset statistics

Two things a reviewer should see before any model runs: the class balance, and the fact that the test split is not drawn from the same distribution as the training split.

In [ ]:
binary = lambda s: np.where(s == 'normal', 'normal', 'attack')

balance = pd.DataFrame({
    'train': pd.Series(binary(train_df['attack'])).value_counts(normalize=True),
    'test': pd.Series(binary(test_df['attack'])).value_counts(normalize=True),
}).mul(100).round(2)
balance.columns = ['train %', 'test %']
print('Class balance')
print(balance.to_string())
print()

print('Ten most frequent labels in train')
print(train_df['attack'].value_counts().head(10).to_string())

### The test split is the point of the benchmark

NSL-KDD's test set deliberately contains attack types that never appear in training. That is what makes it a test of *detecting novel attacks* rather than of memorisation, and it is the main reason recall lands well below precision later on.

Re-splitting the union of the two files would destroy this property, so the published split is preserved as-is.

In [ ]:
train_attacks = set(train_df['attack'].unique())
test_attacks = set(test_df['attack'].unique())
unseen = sorted(test_attacks - train_attacks)

print(f'{len(train_attacks)} attack labels in train, {len(test_attacks)} in test')
print(f'{len(unseen)} appear ONLY in test:')
print('   ', ', '.join(unseen))

unseen_rows = test_df['attack'].isin(unseen).sum()
print(f'\n{unseen_rows:,} test rows ({unseen_rows / len(test_df):.1%}) are of an unseen type.')

## 3. Preprocess

One-hot encode the three nominal columns, standardise the 38 numeric ones, binarise the label to `0 = normal` / `1 = attack`. Every transformer is fitted on **train only** and then applied to test.

The `difficulty` column is dataset metadata, not a network feature — it is dropped. Feeding it to the model would leak.

In [ ]:
X_train_full, y_train_full, X_test_full, y_test_full = load_and_preprocess_nsl_kdd()

print(f'X_train {X_train_full.shape}    X_test {X_test_full.shape}')
print(f'122 features = 38 numeric + 84 one-hot levels (protocol_type, service, flag)')
print(f'attack rate: train {y_train_full.mean():.2%}, test {y_test_full.mean():.2%}')

## 4. Subsample to one TabPFN context

TabPFN v2 accepts at most 10,000 in-context training samples. The baseline therefore has to throw away most of NSL-KDD's 125,973 training rows — it is a hard architectural ceiling, not a tuning choice.

**Lifting that ceiling is exactly what Enhancement 1 (notebook 02) does.**

Both subsamples are stratified so the class balance survives.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit


def stratified_subset(X, y, n, seed=config.SEED):
    """Take a class-balance-preserving subset of n rows."""
    idx, _ = next(StratifiedShuffleSplit(
        n_splits=1, train_size=n, random_state=seed).split(X, y))
    return X[idx], y[idx]


X_train, y_train = stratified_subset(X_train_full, y_train_full, CONTEXT_ROWS)
X_test, y_test = stratified_subset(X_test_full, y_test_full, TEST_ROWS)

discarded = 1 - CONTEXT_ROWS / len(y_train_full)
print(f'context   : {X_train.shape[0]:,} rows  (attack rate {y_train.mean():.2%})')
print(f'test      : {X_test.shape[0]:,} rows  (attack rate {y_test.mean():.2%})')
print(f'discarded : {discarded:.1%} of the available training data')

## 5. Fit and predict

`fit` only caches the context — TabPFN is a prior-fitted network, so there is no gradient training and it returns in well under a second. Essentially all the compute is in `predict`.

In [ ]:
model = TabPFNWrapper(
    random_state=config.SEED,
    n_estimators=N_ESTIMATORS,
    predict_batch_size=1_000,
)

model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test)
y_pred = np.argmax(y_proba, axis=1)

print(f'fit     {model.fit_seconds:6.2f}s   (context caching only)')
print(f'predict {model.predict_seconds:6.1f}s   ({model.predict_seconds / len(y_test) * 1000:.0f}s per 1,000 test rows)')

## 6. Results

In [ ]:
metrics = compute_metrics(y_test, y_pred, y_proba)
print(format_metrics(metrics, title=f'NSL-KDD baseline — demo scale ({TEST_ROWS} test rows)'))

### How this compares to the full-scale runs

The recorded results below are the ones quoted in the report: three seeds, 10,000-row context, 5,000 test rows. The demo run above uses less data in both directions, so expect it to land near them but not on them.

In [ ]:
from tabpfn_nids.evaluation import load_results

recorded = pd.DataFrame(load_results('baseline'))
score_cols = ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']

if len(recorded):
    recorded[score_cols] = recorded[score_cols].astype(float)
    full = recorded[score_cols].agg(['mean', 'std']).T
    comparison = pd.DataFrame({
        'this notebook (demo scale)': [metrics[c] for c in score_cols],
        'recorded mean (3 seeds, full scale)': full['mean'].values,
        'recorded std': full['std'].values,
    }, index=score_cols).round(4)
    print(comparison.to_string())
    print(f"\nRecorded runs used {recorded['context_rows'].iloc[0]} context rows "
          f"and {recorded['test_rows'].iloc[0]} test rows.")
else:
    print('No baseline CSVs in reports/ — run scripts/run_baseline.py first.')

### Reading this honestly

**Precision is high, recall is low.** The model is usually right when it flags an attack, but it misses a large share of them. That is the expected consequence of the unseen attack types quantified in section 2 — the model cannot recognise what was never in its context.

**ROC-AUC near 0.95 alongside F1 near 0.75** says the *ranking* is good and the default 0.5 decision threshold is simply placed badly for this class balance. The precision-recall curve below makes that visible: there are threshold choices with a much better recall trade-off. Threshold tuning is listed in `docs/FUTURE_WORK.md`; it is deliberately not done here, because tuning a threshold on the test set would be the same leak as keeping `difficulty`.

## 7. Figures

In [ ]:
from IPython.display import Image, display

paths = plot_all(y_test, y_proba, metrics['confusion_matrix'], prefix='nb01_baseline')
for name, path in paths.items():
    display(Image(str(path)))

print(f'Notebook runtime: {time.perf_counter() - notebook_started:.0f}s')